<a href="https://colab.research.google.com/github/jaeyong3126/SHIELD_DOC/blob/data_preprocessing/total_datasets_preprocessed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install konlpy pypdf python-docx

In [4]:
%cd /content/drive/MyDrive/SHIELD_DOC
!git pull

/content/drive/MyDrive/SHIELD_DOC
^C


In [5]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 95.6 MB/s eta 0:00:00


In [12]:
import sys
import os
import pandas as pd
import re
from tqdm import tqdm
from collections import Counter
from konlpy.tag import Okt
import importlib

# 환경 설정 및 모듈 불러오기

sys.path.append('/content/drive/MyDrive/SHIELD_DOC')

try:
    from tools.parser import parse_document
    from tools.pii_detector import detect_pii
except ImportError as e:
    print(f"모듈 로드 실패: {e}. 경로를 확인해 주세요.")

# 문서 파싱 및 기초 데이터셋 병합 (기밀 + 정상)

print("문서 파싱 및 기초 데이터셋 병합 시작...")

def parse_folder_to_list(folder_path, label):
    data_list = []
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                parsed_data = parse_document(file_path)
                if parsed_data["text"].strip():
                    data_list.append({
                        "filename": parsed_data["filename"],
                        "text": parsed_data["text"],
                        "label": label
                    })
            except Exception as e:
                print(f"파싱 실패 ({filename}): {e}")
    return data_list

# 기밀(1) 및 정상(0) 문서 파싱
confidential_dir = '/content/drive/MyDrive/Confidential/'
normal_dir = '/content/drive/MyDrive/Normal/'

conf_list = parse_folder_to_list(confidential_dir, 1)
norm_list = parse_folder_to_list(normal_dir, 0)

# 두 데이터 합치기 (Concat)
df_datasets = pd.concat([pd.DataFrame(conf_list), pd.DataFrame(norm_list)], ignore_index=True)
print(f"(총 {len(df_datasets)}건: 기밀 {len(conf_list)}건 / 정상 {len(norm_list)}건)")


# 특수문자 정제 및 PII 마스킹

print("1차 텍스트 정제 및 개인정보(PII) 마스킹 시작...")

def clean_text_basic(text):
    if pd.isna(text) or not text: return ''
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\[\]_]', ' ', str(text))
    return re.sub(r'\s+', ' ', text).strip()

processed_data = []
for index, row in tqdm(df_datasets.iterrows(), total=len(df_datasets), desc="마스킹 및 정제"):
    original_text = row['text']
    pii_result = detect_pii(original_text) # 개인정보 마스킹
    masked_text = pii_result.get("masked_text", original_text)
    processed_data.append(clean_text_basic(masked_text))

df_datasets['cleaned_text'] = processed_data

# 결측치 1차 제거
df_clean = df_datasets[df_datasets['cleaned_text'] != ''].copy()


# 중복 데이터 제거

before_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['cleaned_text'], keep='first')
df_clean.reset_index(drop=True, inplace=True)
after_count = len(df_clean)

print(f"{before_count}건 ➔ {after_count}건 (삭제 {before_count - after_count}건)")


# EDA 분석 (단어 빈도수 및 길이 분포)

print("EDA (불용어 탐색 및 데이터 분포 확인)")

okt = Okt()
confidential_nouns, normal_nouns = [], []

for text in tqdm(df_clean[df_clean['label'] == 1]['cleaned_text'], desc="기밀 문서 분석"):
    confidential_nouns.extend([w for w in okt.nouns(str(text)) if len(w) > 1])

for text in tqdm(df_clean[df_clean['label'] == 0]['cleaned_text'], desc="정상 문서 분석"):
    normal_nouns.extend([w for w in okt.nouns(str(text)) if len(w) > 1])

print("[EDA] 가장 많이 등장한 단어 Top 20")
print(f"기밀 문서: {Counter(confidential_nouns).most_common(20)}")
print(f"정상 문서: {Counter(normal_nouns).most_common(20)}")

# 길이 분포 간단 요약
df_clean['word_count'] = df_clean['cleaned_text'].astype(str).apply(lambda x: len(x.split()))
print("[데이터 길이 요약]")
display(df_clean.groupby('label')['word_count'].describe().round(1))


# 토큰화, 소문자화, 숫자제거

standard_stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다','입니다','있습니다','합니다','됨','음','수','것','및','등','본','이','있','하','그','되','보','않','없','나','사람','주','아니','같','우리','때','년','지','대하','오','말','일','그렇','위하','때문','그것','두','말하','알','그러나','받','못하','그런','또','문제','더','사회','많','그리고','좋','크','따르','중','나오','가지','씨','시키','만들','지금','생각하','그러','속','하나','집','살','모르','적','월','데','자신','안','어떤','내','내다','경우','명','생각','시간','그녀','다시','이런','앞','보이','번','다른',
                      '어떻','여자','개','전','사실','이렇','점','싶','정도','원','통하','소리','놓','있다', '되다', '이다', '않다', '되어다', '같다', '그렇다', '많다', '들다', '나다', '받다', '가다',
                      '오다', '알다', '주다', '이런', '저런', '어떤', '빠지다', '걸리다', '드리다']
domain_stopwords = ['한빛', '반도체', '당사', '부서', '문서', '위해', '대상', '내용', '사항', '관련', '참고', '확인', '작성', '이용', '정리', '기준','hanbit', 'semiconductor', 'corporate', 'document', 'example', 'semi', 'ra',
                    'g', 'a', 's', 'nm', 'mcu', 'vip', '가표', '밉다','com', 'co', 'kr', 'net', 'www','hbs', 'hr', 'emg', 'acc', 'wel','늘다', '보다', '따르다', '이번']
stopwords = list(set(standard_stopwords + domain_stopwords))

def extract_and_polish(text):
    if not isinstance(text, str): return ""

    # 형태소 분석 및 불용어 제거
    morphs = okt.pos(text, stem=True)
    words = []
    for word, pos in morphs:
        if pos in ['Noun', 'Verb', 'Adjective', 'Alpha']:
            if word not in stopwords and (len(word) > 1 or pos in ['Alpha']):
                words.append(word)

    # 소문자화 및 숫자 제거
    joined_text = " ".join(words).lower()
    joined_text = re.sub(r'\d+', ' ', joined_text)
    return re.sub(r'\s+', ' ', joined_text).strip()

tqdm.pandas()
df_clean['final_tokens'] = df_clean['cleaned_text'].progress_apply(extract_and_polish)

# 전처리 후 결측치 확인 및 제거
df_clean = df_clean[df_clean['final_tokens'] != '']

# 중복 파일 제거
before_count_1 = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['final_tokens'], keep='first')
df_clean.reset_index(drop=True, inplace=True)
after_count_1 = len(df_clean)
print(f"{before_count_1}건 ➔ {after_count_1}건 (삭제 {before_count_1 - after_count_1}건)")

# 최종 파일 하나로 깔끔하게 저장

print("최종 데이터셋 저장")

# 필요한 핵심 컬럼만 추려서 저장
df_final_output = df_clean[['filename', 'label', 'final_tokens']]
final_path = '/content/drive/MyDrive/Module_proj1/final_datasets.csv'
df_final_output.to_csv(final_path, index=False, encoding='utf-8-sig')

print(f"파일이 성공적으로 저장되었습니다.")
print(f"저장 경로: {final_path}")

display(df_final_output.head(3))

문서 파싱 및 기초 데이터셋 병합 시작...
(총 1860건: 기밀 1193건 / 정상 667건)
1차 텍스트 정제 및 개인정보(PII) 마스킹 시작...


마스킹 및 정제: 100%|██████████| 1860/1860 [02:34<00:00, 12.01it/s]


1860건 ➔ 1860건 (삭제 0건)
EDA (불용어 탐색 및 데이터 분포 확인)


정상 문서 분석: 100%|██████████| 667/667 [00:13<00:00, 48.49it/s]

[EDA] 가장 많이 등장한 단어 Top 20
기밀 문서: [('검토', 3302), ('확인', 2642), ('조건', 2037), ('조정', 1634), ('다음', 1432), ('비교', 1396), ('기준', 1391), ('문서', 1302), ('항목', 1275), ('이번', 1188), ('반영', 1162), ('단가', 1084), ('계약', 1037), ('정리', 1034), ('현재', 1019), ('결과', 1015), ('측정', 1010), ('공급', 998), ('조직', 948), ('반도체', 910)]
정상 문서: [('안내', 1524), ('확인', 1360), ('한빛', 858), ('정리', 825), ('운영', 815), ('반도체', 812), ('사내', 806), ('이용', 667), ('자료', 628), ('변경', 600), ('내용', 555), ('관련', 539), ('업무', 530), ('항목', 530), ('사용', 526), ('공용', 505), ('교육', 486), ('사항', 473), ('시간', 439), ('표시', 420)]
[데이터 길이 요약]


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,667.0,111.3,32.0,40.0,92.5,105.0,148.0,198.0
1,1193.0,157.0,47.6,58.0,119.0,182.0,197.0,244.0


100%|██████████| 1860/1860 [00:42<00:00, 43.51it/s]

1860건 ➔ 1710건 (삭제 150건)
최종 데이터셋 저장
파일이 성공적으로 저장되었습니다.
저장 경로: /content/drive/MyDrive/Module_proj1/final_datasets.csv


,filename,label,final_tokens
0,pii_jeh_036.txt,1,사내 보안등 대외 서명 신규 배치 인적 직원 명부 번호 hbs hr 조직 개편 배치...
1,pii_jeh_020.txt,1,사업 안전 보안 비상 연락망 번호 hbs emg 최종 정일 관리 비상 연락망 사업 ...
2,pii_jeh_023.txt,1,캠퍼스 외부 협력 출입 신청서 신청 번호 hbs acc 접수 일자 관리 사업 출입 ...
